In [7]:
RUN_ID = "9c748fd62c3e440f8f2a6bb7145d83ce"

In [8]:
import json
import mlflow
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.gridspec as gridspec

ORIGINAL_PROJECTS = {
    "PyCQA/pylint", "Qiskit/qiskit-terra", "apache/airflow",
    "h2oai/h2o-3", "oblac/jodd", "orientechnologies/orientdb",
    "real-logic/aeron", "spring-projects/spring-security",
    "vaexio/vaex", "wildfly/wildfly",
}

# -- language map from dataset -------------------------------------------------
proj_lang = {}
with open("dataset/v3/test.jsonl") as f:
    for line in f:
        r = json.loads(line)
        proj_lang[r["repo"]] = r["language"]

# -- metrics from MLflow -------------------------------------------------------
client   = mlflow.tracking.MlflowClient()
run      = client.get_run(RUN_ID)
run_name = run.info.run_name

data = []
for key, value in run.data.metrics.items():
    if not key.endswith("/bleu"):
        continue
    project = key[: -len("/bleu")]
    data.append({
        "project":    project,
        "short_name": project.split("/")[-1],
        "bleu":       value,
        "era":        "pre-2024"  if project in ORIGINAL_PROJECTS else "post-2024",
        "language":   proj_lang.get(project, "unknown"),
    })

import pandas as pd
df = pd.DataFrame(data)
print(df.groupby(["language", "era"])["bleu"].describe().round(2))

                    count   mean   std    min    25%    50%    75%    max
language era                                                             
java     post-2024    5.0  19.32  4.42  12.93  18.14  18.66  22.63  24.25
         pre-2024     5.0  17.09  1.93  15.02  16.08  16.38  18.05  19.94
python   post-2024    5.0  18.20  4.28  14.18  14.95  16.42  21.61  23.85
         pre-2024     5.0  17.43  3.03  13.91  15.82  16.19  20.43  20.78


In [ ]:
plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "font.size":         9.5,
    "axes.linewidth":    0.6,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size":  2,
    "ytick.major.size":  2,
})

COLORS = {
    ("pre-2024",  "python"): "#4477AA",
    ("pre-2024",  "java"):   "#114466",
    ("post-2024", "python"): "#EE7733",
    ("post-2024", "java"):   "#994411",
}
BG     = "white"
MID_BG = "#F2F2F2"
YMAX   = df["bleu"].max() * 1.16

BAR_W   = 0.55
SPACING = 0.80
MARGIN  = 0.40

def xs_for(n):
    return np.arange(n) * SPACING

def xlim_for(n):
    return (-MARGIN, (n - 1) * SPACING + MARGIN)

def data_range(n):
    lo, hi = xlim_for(n)
    return hi - lo

fig = plt.figure(figsize=(15, 11), facecolor=BG)
fig.suptitle(
    f"BLEU score per project  ·  {run_name}",
    fontsize=11, fontweight="bold", y=0.99, color="#111111",
)

outer = gridspec.GridSpec(
    2, 1, hspace=0.45,
    left=0.07, right=0.99, top=0.95, bottom=0.10,
)

for row_i, (lang, lang_label) in enumerate(zip(["python", "java"], ["Python", "Java"])):
    sub  = df[df["language"] == lang]
    pre  = sub[sub["era"] == "pre-2024" ].sort_values("bleu", ascending=False)
    post = sub[sub["era"] == "post-2024"].sort_values("bleu", ascending=False)
    pre_mean  = pre["bleu"].mean()
    post_mean = post["bleu"].mean()
    n = max(len(pre), len(post))

    wr_proj = data_range(n)
    wr_avg  = data_range(2)
    inner = gridspec.GridSpecFromSubplotSpec(
        1, 3, subplot_spec=outer[row_i],
        width_ratios=[wr_proj, wr_avg, wr_proj], wspace=0.03,
    )
    share   = fig.axes[0] if fig.axes else None
    ax_pre  = fig.add_subplot(inner[0], sharey=share)
    ax_mid  = fig.add_subplot(inner[1], sharey=ax_pre)
    ax_post = fig.add_subplot(inner[2], sharey=ax_pre)

    c_pre  = COLORS[("pre-2024",  lang)]
    c_post = COLORS[("post-2024", lang)]

    def style(ax, mid=False):
        ax.set_facecolor(MID_BG if mid else BG)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_color("#bbbbbb")
        ax.spines["bottom"].set_color("#bbbbbb")
        ax.yaxis.set_major_locator(ticker.MultipleLocator(5))
        ax.grid(axis="y", which="major", color="#e4e4e4", linewidth=0.6, zorder=0)
        ax.set_ylim(0, YMAX)
        ax.tick_params(axis="x", length=0, labelsize=8.5)
        ax.tick_params(axis="y", labelsize=8, color="#aaaaaa")
        if mid:
            ax.spines["left"].set_visible(False)
            ax.tick_params(axis="y", left=False, labelleft=False)

    def val_label(ax, x, y, fs=8):
        ax.text(x, y + 0.25, f"{y:.1f}",
                ha="center", va="bottom", fontsize=fs, color="#222222")

    style(ax_pre)
    xs = xs_for(len(pre))
    ax_pre.bar(xs, pre["bleu"].values, width=BAR_W,
               color=c_pre, edgecolor="white", linewidth=0.4, zorder=3)
    for x, v in zip(xs, pre["bleu"].values):
        val_label(ax_pre, x, v)
    ax_pre.set_xticks(xs)
    ax_pre.set_xticklabels(pre["short_name"].values, rotation=38, ha="right")
    ax_pre.set_xlim(*xlim_for(len(pre)))
    ax_pre.set_ylabel("BLEU", fontsize=9.5, labelpad=3)
    ax_pre.text(-0.12, 0.5, lang_label, transform=ax_pre.transAxes,
                fontsize=10, fontweight="bold", color="#333333",
                va="center", ha="right", rotation=90)
    if row_i == 0:
        ax_pre.set_title("pre-2024 (original)", fontsize=9.5,
                         fontweight="bold", color=c_pre, pad=4)

    style(ax_mid, mid=True)
    xs_avg = xs_for(2)
    ax_mid.bar(xs_avg[0], pre_mean,  width=BAR_W, color=c_pre,
               edgecolor="white", linewidth=0.4, zorder=3)
    ax_mid.bar(xs_avg[1], post_mean, width=BAR_W, color=c_post,
               edgecolor="white", linewidth=0.4, zorder=3)
    val_label(ax_mid, xs_avg[0], pre_mean,  fs=9)
    val_label(ax_mid, xs_avg[1], post_mean, fs=9)
    ax_mid.set_xticks(xs_avg)
    ax_mid.set_xticklabels(["pre\navg", "post\navg"], fontsize=8.5)
    ax_mid.set_xlim(*xlim_for(2))
    if row_i == 0:
        ax_mid.set_title("Avg", fontsize=9.5, fontweight="bold",
                         color="#555555", pad=4)

    style(ax_post)
    ax_post.spines["left"].set_visible(False)
    ax_post.tick_params(axis="y", left=False, labelleft=False)
    xs = xs_for(len(post))
    ax_post.bar(xs, post["bleu"].values, width=BAR_W,
                color=c_post, edgecolor="white", linewidth=0.4, zorder=3)
    for x, v in zip(xs, post["bleu"].values):
        val_label(ax_post, x, v)
    ax_post.set_xticks(xs)
    ax_post.set_xticklabels(post["short_name"].values, rotation=38, ha="right")
    ax_post.set_xlim(*xlim_for(len(post)))
    if row_i == 0:
        ax_post.set_title("post-2024 (new)", fontsize=9.5,
                          fontweight="bold", color=c_post, pad=4)

    for ax in (ax_pre, ax_mid):
        ax.axvline(ax.get_xlim()[1], color="#cccccc", lw=0.7, zorder=10)

plt.show()